In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor

fixed_params = {
    "colsample_bytree": 0.7728434442477368,
    "learning_rate": 0.0035099451126927843,
    "max_bin": 164,
    "max_depth": 15,
    "min_child_samples": 12,
    "min_split_gain": 0.0006723396861006931,
    "n_estimators": 1161,
    "num_leaves": 177,
    "reg_alpha": 0.0010103063701743973,
    "reg_lambda": 0.002652385528961019,
    "subsample": 0.9757392798329956,
    "subsample_freq": 2
}
n_components = 4
seed_list = [500, 530, 590, 710, 790, 1030, 1040, 1060, 1090, 1100, 1300, 1330, 1360, 1440, 1500, 1540, 1550, 1710, 1750, 1830, 1860, 1980, 2000]

base_dir = Path.cwd()
spectra_path = base_dir / "pattern.xlsx"
label_path = base_dir / "label-1.xlsx"

output_path = spectra_path.with_name("LightGBM_SeedStabilityResult.xlsx")

spectra_df = pd.read_excel(spectra_path, header=None)
label_df = pd.read_excel(label_path, header=None)

id_col = "ID"
target_col = "HA Yield"
feature_cols = [f"Feature_{i}" for i in range(1, spectra_df.shape[1])]
spectra_df.columns = [id_col] + feature_cols
label_df = label_df.iloc[:, :2].copy()
label_df.columns = [id_col, target_col]

def normalize_id(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)):
        return str(int(x)) if x.is_integer() else str(x).rstrip("0").rstrip(".")
    return str(x).strip()

spectra_df[id_col] = spectra_df[id_col].map(normalize_id)
label_df[id_col] = label_df[id_col].map(normalize_id)
label_df[target_col] = pd.to_numeric(label_df[target_col], errors="coerce")

label_check = label_df.groupby(id_col)[target_col].nunique(dropna=False)
conflict_ids = label_check[label_check > 1].index.tolist()
if conflict_ids:
    raise ValueError(f"The following IDs have multiple different HA yields, please check label.xlsx: {conflict_ids}")

label_unique = label_df.drop_duplicates(subset=[id_col], keep="first")
df = spectra_df.merge(label_unique, on=id_col, how="left")

if df[target_col].isna().any():
    missing_ids = df.loc[df[target_col].isna(), id_col].unique()
    raise ValueError(f"The following IDs are not found in label.xlsx: {missing_ids}")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols if col != target_col]
X = df[feature_cols].astype(float)
y = df[target_col].astype(float)

X_standard = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0).replace(0, 1)

max_components = min(X_standard.shape[0] - 1, X_standard.shape[1])
if n_components > max_components:
    raise ValueError(f"At most {max_components} PLS components can be extracted from the current data, cannot set to {n_components}.")

X_pls_input = X_standard.values
y_pls_input = y.values.reshape(-1, 1)
pls = PLSRegression(n_components=n_components, scale=False)
pls.fit(X_pls_input, y_pls_input)

X_scores = pls.x_scores_
X_loadings = pls.x_loadings_
X_reconstructed = X_scores @ X_loadings.T
ss_total = np.sum(X_pls_input ** 2)
ss_residual = np.sum((X_pls_input - X_reconstructed) ** 2)
cumulative_variance_ratio = 1 - ss_residual / ss_total
pls_cols = [f"PLS_Component_{i + 1}" for i in range(n_components)]

df_pls = pd.DataFrame(X_scores, columns=pls_cols)
df_pls.insert(0, id_col, df[id_col].values)
df_pls[target_col] = y.values
X_model = df_pls[pls_cols].astype(float)
y_model = df_pls[target_col].astype(float)
groups = df_pls[id_col].astype(str)

seed_result_list = []

for seed in seed_list:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(splitter.split(X_model, y_model, groups=groups))
    X_train = X_model.iloc[train_idx]
    X_test = X_model.iloc[test_idx]
    y_train = y_model.iloc[train_idx]
    y_test = y_model.iloc[test_idx]

    model = LGBMRegressor(
        objective="regression",
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
        **fixed_params
    )
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mre = np.mean(np.abs(y_train - y_train_pred) / y_train) * 100
    train_r2 = r2_score(y_train, y_train_pred)

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mre = np.mean(np.abs(y_test - y_test_pred) / y_test) * 100
    test_r2 = r2_score(y_test, y_test_pred)

    seed_row = {
        "RandomSeed": seed,
        "Train_MSE": train_mse,
        "Train_MAE": train_mae,
        "Train_MRE(%)": train_mre,
        "Train_R2": train_r2,
        "Test_MSE": test_mse,
        "Test_MAE": test_mae,
        "Test_MRE(%)": test_mre,
        "Test_R2": test_r2
    }
    seed_result_list.append(seed_row)

seed_summary_df = pd.DataFrame(seed_result_list)

stat_cols = [c for c in seed_summary_df.columns if c != "RandomSeed"]
stat_summary = pd.DataFrame({
    "Metric": stat_cols,
    "Mean": [seed_summary_df[col].mean() for col in stat_cols],
    "Std": [seed_summary_df[col].std() for col in stat_cols],
    "CV(%)": [(seed_summary_df[col].std() / seed_summary_df[col].mean()) * 100 for col in stat_cols]
})

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_pls.to_excel(writer, sheet_name="pls_scores_4_components", index=False)
    seed_summary_df.to_excel(writer, sheet_name="all_seed_metrics", index=False)
    stat_summary.to_excel(writer, sheet_name="seed_stat_mean_std", index=False)
    pd.DataFrame(list(fixed_params.items()), columns=["Parameter", "Fixed Value"]).to_excel(writer, sheet_name="fixed_lgb_params", index=False)

print("\n==================== Global PLS Info ====================")
print(f"Fixed PLS Components: {n_components}")
print(f"PLS Cumulative Variance Ratio: {cumulative_variance_ratio:.4%}")
print(f"Total seed runs: {len(seed_list)}")
print("\n==================== Average Performance Across All Seeds ====================")
print(f"Average Test R2: {seed_summary_df['Test_R2'].mean():.4f} +/- {seed_summary_df['Test_R2'].std():.4f}")
print(f"Average Train R2: {seed_summary_df['Train_R2'].mean():.4f} +/- {seed_summary_df['Train_R2'].std():.4f}")
print(f"\nAll seed stability results saved to: {output_path}")


==================== Global PLS Info ====================
Fixed PLS Components: 4
PLS Cumulative Variance Ratio: 97.2069%
Total seed runs: 23

==================== Average Performance Across All Seeds ====================
Average Test R2: 0.7309 +/- 0.0623
Average Train R2: 0.8237 +/- 0.0312

All seed stability results saved to: E:\YRH-HSI\python data\MUST\must\LightGBM_SeedStabilityResult.xlsx


In [5]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings("ignore")

fixed_params = {
    'n_estimators': 1000,
    'max_depth': 9,
    'max_features': 0.5,
    'min_samples_split': 3,
    'min_samples_leaf': 2,
    'max_samples': 0.8,
    'bootstrap': True,
    'ccp_alpha': 0.003
}
n_components = 4
seed_list = [500, 530, 590, 710, 790, 1030, 1040, 1060, 1090, 1100, 1300, 1330, 1360, 1440, 1500, 1540, 1550, 1710, 1750, 1830, 1860, 1980, 2000]

base_dir = Path.cwd()
spectra_path = base_dir / "pattern.xlsx"
label_path = base_dir / "label-1.xlsx"

output_path = spectra_path.with_name("RF_SeedStabilityResult.xlsx")

spectra_df = pd.read_excel(spectra_path, header=None)
label_df = pd.read_excel(label_path, header=None)

id_col = "ID"
target_col = "HA Yield"
feature_cols = [f"Feature_{i}" for i in range(1, spectra_df.shape[1])]
spectra_df.columns = [id_col] + feature_cols
label_df = label_df.iloc[:, :2].copy()
label_df.columns = [id_col, target_col]

def normalize_id(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return str(x)
    if isinstance(x, (float, np.floating)):
        return str(int(x)) if x.is_integer() else str(x).rstrip("0").rstrip(".")
    return str(x).strip()

spectra_df[id_col] = spectra_df[id_col].map(normalize_id)
label_df[id_col] = label_df[id_col].map(normalize_id)
label_df[target_col] = pd.to_numeric(label_df[target_col], errors="coerce")

label_unique = label_df.drop_duplicates(subset=[id_col], keep="first")
df = spectra_df.merge(label_unique, on=id_col, how="left")

X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean()).fillna(0)
y = df[target_col].astype(float)

X_standard = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0).replace(0, 1)

max_components = min(X_standard.shape[0] - 1, X_standard.shape[1])
pls = PLSRegression(n_components=n_components, scale=False)
pls.fit(X_standard.values, y.values.reshape(-1, 1))
X_scores = pls.x_scores_
X_reconstructed = X_scores @ pls.x_loadings_.T
ss_total = np.sum(X_standard.values ** 2)
ss_residual = np.sum((X_standard.values - X_reconstructed) ** 2)
cumulative_variance_ratio = 1 - ss_residual / ss_total

pls_cols = [f"PLS_Component_{i + 1}" for i in range(n_components)]
df_pls = pd.DataFrame(X_scores, columns=pls_cols)
df_pls.insert(0, id_col, df[id_col].values)
df_pls[target_col] = y.values
X_model = df_pls[pls_cols].astype(float)
y_model = df_pls[target_col].astype(float)
groups = df_pls[id_col].astype(str)

seed_result_list = []

for seed in seed_list:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(splitter.split(X_model, y_model, groups=groups))
    X_train = X_model.iloc[train_idx]
    X_test = X_model.iloc[test_idx]
    y_train = y_model.iloc[train_idx]
    y_test = y_model.iloc[test_idx]

    model = RandomForestRegressor(
        random_state=seed,
        n_jobs=-1,
        **fixed_params
    )
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mre = np.mean(np.abs(y_train - y_train_pred) / y_train) * 100
    train_r2 = r2_score(y_train, y_train_pred)

    test_mse = mean_squared_error(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mre = np.mean(np.abs(y_test - y_test_pred) / y_test) * 100
    test_r2 = r2_score(y_test, y_test_pred)

    seed_row = {
        "RandomSeed": seed,
        "Train_MSE": train_mse,
        "Train_MAE": train_mae,
        "Train_MRE(%)": train_mre,
        "Train_R2": train_r2,
        "Test_MSE": test_mse,
        "Test_MAE": test_mae,
        "Test_MRE(%)": test_mre,
        "Test_R2": test_r2
    }
    seed_result_list.append(seed_row)

seed_summary_df = pd.DataFrame(seed_result_list)

stat_cols = [c for c in seed_summary_df.columns if c != "RandomSeed"]
stat_summary = pd.DataFrame({
    "Metric": stat_cols,
    "Mean": [seed_summary_df[col].mean() for col in stat_cols],
    "Std": [seed_summary_df[col].std() for col in stat_cols],
    "CV(%)": [(seed_summary_df[col].std() / seed_summary_df[col].mean()) * 100 for col in stat_cols]
})

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_pls.to_excel(writer, sheet_name="pls_scores_4_components", index=False)
    seed_summary_df.to_excel(writer, sheet_name="all_seed_metrics", index=False)
    stat_summary.to_excel(writer, sheet_name="seed_stat_mean_std", index=False)
    pd.DataFrame(list(fixed_params.items()), columns=["Parameter", "Fixed Value"]).to_excel(writer, sheet_name="fixed_rf_params", index=False)

print("\n==================== Global PLS Info ====================")
print(f"Fixed PLS Components: {n_components}")
print(f"PLS Cumulative Variance Ratio: {cumulative_variance_ratio:.4%}")
print(f"Total seed runs: {len(seed_list)}")
print("\n==================== Average Performance Across All Seeds ====================")
print(f"Average Test R2: {seed_summary_df['Test_R2'].mean():.4f} +/- {seed_summary_df['Test_R2'].std():.4f}")
print(f"Average Train R2: {seed_summary_df['Train_R2'].mean():.4f} +/- {seed_summary_df['Train_R2'].std():.4f}")
print(f"\nAll seed stability results saved to: {output_path}")


==================== Global PLS Info ====================
Fixed PLS Components: 4
PLS Cumulative Variance Ratio: 97.2069%
Total seed runs: 23

==================== Average Performance Across All Seeds ====================
Average Test R2: 0.7835 +/- 0.0380
Average Train R2: 0.8715 +/- 0.0099

All seed stability results saved to: E:\YRH-HSI\python data\MUST\must\RF_SeedStabilityResult.xlsx
